# Vessel dataset exploration and identity resolution

This notebook explores the case study CSV and runs vessel identification snippets.

In [ ]:
import sys
import os
import pandas as pd

REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO_ROOT)

from scripts.imo_validation import is_valid_imo
from scripts.vessel_identity import flag_invalid_records, same_vessel, vessel_timeline

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, 'case_study_dataset_202509152039.csv')
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
print('Shape:', df.shape)
print('Columns:', list(df.columns[:15]), '...')

## IMO validation

Valid IMO: 7 digits, last digit = (d1×7 + d2×6 + … + d6×2) mod 10.

In [ ]:
imos = df['imo'].dropna().astype(int)
valid = imos.apply(is_valid_imo)
print('Valid IMO count:', valid.sum())
print('Invalid IMO count:', (~valid).sum())
print('Sample invalid IMOs:', imos[~valid].unique()[:10].tolist())

## Flag invalid and conflicting records

In [ ]:
df_flagged = flag_invalid_records(df)
print('Invalid IMO rows:', df_flagged['invalid_imo'].sum())
print('IMO–MMSI conflict rows:', df_flagged['imo_mmsi_conflict'].sum())
print('Duplicate (imo, mmsi) rows:', df_flagged['duplicate_key'].sum())

## Same-vessel check and timeline

Example: HOEGH TROTTER (IMO 9710749) has multiple MMSIs over time.

In [ ]:
example_imo = 9710749
timeline = vessel_timeline(df, example_imo)
for r in timeline[:5]:
    print(r)

r1 = df[df['imo'] == example_imo].iloc[0].to_dict()
r2 = df[df['imo'] == example_imo].iloc[-1].to_dict()
print('\nsame_vessel(r1, r2):', same_vessel(r1, r2))

## Conversational AI: caching and session (sketch)

Run the sketch to see cache hit on repeated filters and session merge on refinement.

In [ ]:
from scripts.conversational_ai_sketch import handle_search_turn

sid = 'nb-session'
r1 = handle_search_turn(sid, {'vessel_type': 'Container', 'flag': 'SG'}, df, limit=5)
print('Turn 1:', len(r1['result']), 'vessels', 'from_cache:', r1['from_cache'])
r2 = handle_search_turn(sid, {'vessel_type': 'Container', 'flag': 'SG'}, df, limit=5)
print('Turn 2 (same):', len(r2['result']), 'from_cache:', r2['from_cache'])
r3 = handle_search_turn(sid, {'builtYear_min': 2020}, df, limit=5)
print('Turn 3 (refine):', len(r3['result']), 'filters:', r3['filters'])